# Python Finance Paper Replication: Empirical Asset Pricing via ML, GKX

# Step 1: Let's build the 94 characteristics

## 1) Compustat/CRSP variables

According to GHZ SAS code and their literature.

### a) Annual Compustat Data

Get: sale, revt, cogs, xsga, dp, xrd, xad, ib, ebitda, ebit, nopi, spi, pi, txp, ni, txfed, txfo, txt, xint, capx, oancf, dvt, ob, gdwlia, gdwlip, gwo, rect, act, che, ppegt, invt, at, aco, intan, ao, ppent, gdwl, fatb, fatl, lct, dlc, dltt, lt, dm, dcvt, cshrc, dcpstk, pstk, ap, lco, lo, drc, drlt, txdi, ceq, scstkc, emp, csho, sic, naics, prcc_f, fyear, datadate, gvkey

### b) Quaterly Compustat Data

Get = ibq, saleq, txtq, revtq, cogsq, xsgaq, atq, actq, cheq, lctq, dlcq, ppentq, prccq, cshoq, seqq, pstkq, atq, ltq, pstkrq, fyearq, fqtr, rdq, sic, ceqq, gvkey, datadate

### c) Monthly CRSP Stock File

Get = permno, mthcaldt, mthret, mthprc, shrout, mthvol

### d) CRSP Delisting Information

Get = delistingdt, delret, delretmisstype, delactiontype, delstatustype, delreasontype

## 2) Use SQL to get the data

Will use SQL to pull data. Could use the Tidy Finance library (https://www.tidy-finance.org/chapters/wrds-crsp-and-compustat.html) which uses SQL too.

In [ ]:
# Connect to WRDS

import pandas as pd
import wrds

db = wrds.Connection(wrds_username='hcdb') 

Loading library list...
Done


In [18]:
# Get the data using SQL

funda = db.raw_sql("""
    SELECT c.gvkey, f.datadate, f.fyear, c.cik, SUBSTR(c.sic, 1, 2) AS sic2,
           c.sic, c.naics,
           f.sale, f.revt, f.cogs, f.xsga, f.dp, f.xrd, f.xad, f.ib, f.ebitda,
           f.ebit, f.nopi, f.spi, f.pi, f.txp, f.ni, f.txfed, f.txfo, f.txt, f.xint,
           f.capx, f.oancf, f.dvt, f.ob, f.gdwlia, f.gdwlip, f.gwo,
           f.rect, f.act, f.che, f.ppegt, f.invt, f.at, f.aco, f.intan, f.ao,
           f.ppent, f.gdwl, f.fatb, f.fatl,
           f.lct, f.dlc, f.dltt, f.lt, f.dm, f.dcvt, f.cshrc, f.dcpstk, f.pstk,
           f.ap, f.lco, f.lo, f.drc, f.drlt, f.txdi,
           f.ceq, f.scstkc, f.emp, f.csho,
           ABS(f.prcc_f) AS prcc_f,
           f.csho * ABS(f.prcc_f) AS mve_f
    FROM comp.funda AS f
    JOIN comp.company AS c
      ON f.gvkey = c.gvkey
    WHERE f.at IS NOT NULL
      AND f.prcc_f IS NOT NULL
      AND f.ni IS NOT NULL
      AND f.datadate >= '1975-01-01'
      AND f.indfmt = 'INDL'
      AND f.datafmt = 'STD'
      AND f.popsrc = 'D'
      AND f.consol = 'C'
""", date_cols=['datadate'])

# note that we are pulling from two diffe

## 3) Built the 94 characteristics